# Consultas directas sobre Parquet

In [6]:
from pathlib import Path
import os
import duckdb
from IPython.display import display, Markdown
BASES = [Path.cwd(), *Path.cwd().parents]
CANDIDATOS = (p for base in BASES for p in (base, base/'persona-b', base/'Persona_B_Exploracion_EDA'))
ROOT = next((p for p in CANDIDATOS if (p/'sql/eda/00_vista.sql').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f'No encuentro sql/eda/00_vista.sql desde {Path.cwd()}. Extrae el contenido del ZIP en la raíz del repositorio o indica ROOT = Path(\"/ruta/a/tu/repositorio\").')
if not list((ROOT/'data/raw').glob('*/*/*.parquet')):
    raise FileNotFoundError(f'En {ROOT}/data/raw no hay Parquet bajo */*/*.parquet. Coloca sql/, notebooks/ y docs/ en la raíz del fork y ejecuta python scripts/download_data.py allí.')

os.chdir(ROOT)
print('Raíz detectada:', ROOT)
con = duckdb.connect(':memory:')
print('DuckDB', duckdb.__version__)
def tabla(columnas,filas):
    esc = lambda x: str('NULL' if x is None else x).replace('|','\\|').replace('\n',' ')
    return '\n'.join(['| '+' | '.join(map(esc,columnas))+' |','| '+' | '.join(['---']*len(columnas))+' |']+['| '+' | '.join(map(esc,f))+' |' for f in filas])
def ejecutar(ruta):
    sql = (ROOT/ruta).read_text()
    salidas=[]
    for sentencia in (x.strip() for x in sql.split(';') if x.strip()):
        resultado=con.execute(sentencia)
        if resultado.description:
            columnas=[x[0] for x in resultado.description]
            filas=resultado.fetchall()
            salidas.append((columnas,filas))
            display(Markdown('### '+str(ruta)+'\n'+tabla(columnas,filas[:25])))
    return salidas

Raíz detectada: /Users/Camila/Desktop/CAMILA UNIVERSIDAD/8SEMESTRE/DataScience/Lab8-DS
DuckDB 1.5.6


## Inventario, recuento, columnas, tipos y muestra

In [9]:
resultados={}
for f in sorted((ROOT/'sql/exploration').glob('0[1-4]_*.sql')):
    resultados[f.name]=ejecutar(f.relative_to(ROOT))

### sql/exploration/01_archivos.sql
| tipo | anio_archivo | archivos | filas_footer | grupos_filas |
| --- | --- | --- | --- | --- |
| green | 2024 | 12 | 660218 | 12 |
| yellow | 2024 | 12 | 41169720 | 44 |
| green | 2025 | 12 | 591375 | 12 |
| yellow | 2025 | 12 | 48722602 | 53 |
| green | 2026 | 8 | 337114 | 8 |
| yellow | 2026 | 8 | 29703355 | 32 |

### sql/exploration/02_registros.sql
| tipo | anio_archivo | filas_count |
| --- | --- | --- |
| green | 2024 | 660218 |
| yellow | 2024 | 41169720 |
| green | 2025 | 591375 |
| yellow | 2025 | 48722602 |
| green | 2026 | 337114 |
| yellow | 2026 | 29703355 |

### sql/exploration/03_columnas.sql
| tipo | columna | tipos_parquet | archivos_con_columna |
| --- | --- | --- | --- |
| yellow | Airport_fee | DOUBLE | 32 |
| green | DOLocationID | INT32 | 32 |
| yellow | DOLocationID | INT32 | 32 |
| green | PULocationID | INT32 | 32 |
| yellow | PULocationID | INT32 | 32 |
| green | RatecodeID | INT64 | 32 |
| yellow | RatecodeID | INT64 | 32 |
| green | VendorID | INT32 | 32 |
| yellow | VendorID | INT32 | 32 |
| green | cbd_congestion_fee | DOUBLE | 20 |
| yellow | cbd_congestion_fee | DOUBLE | 20 |
| green | congestion_surcharge | DOUBLE | 32 |
| yellow | congestion_surcharge | DOUBLE | 32 |
| green | ehail_fee | DOUBLE | 32 |
| green | extra | DOUBLE | 32 |
| yellow | extra | DOUBLE | 32 |
| green | fare_amount | DOUBLE | 32 |
| yellow | fare_amount | DOUBLE | 32 |
| green | improvement_surcharge | DOUBLE | 32 |
| yellow | improvement_surcharge | DOUBLE | 32 |
| green | lpep_dropoff_datetime | INT64 | 32 |
| green | lpep_pickup_datetime | INT64 | 32 |
| green | mta_tax | DOUBLE | 32 |
| yellow | mta_tax | DOUBLE | 32 |
| green | passenger_count | INT64 | 32 |

### sql/exploration/04_tipos_y_muestra.sql
| column_name | column_type | null | key | default | extra |
| --- | --- | --- | --- | --- | --- |
| VendorID | INTEGER | YES | NULL | NULL | NULL |
| lpep_pickup_datetime | TIMESTAMP | YES | NULL | NULL | NULL |
| lpep_dropoff_datetime | TIMESTAMP | YES | NULL | NULL | NULL |
| store_and_fwd_flag | VARCHAR | YES | NULL | NULL | NULL |
| RatecodeID | BIGINT | YES | NULL | NULL | NULL |
| PULocationID | INTEGER | YES | NULL | NULL | NULL |
| DOLocationID | INTEGER | YES | NULL | NULL | NULL |
| passenger_count | BIGINT | YES | NULL | NULL | NULL |
| trip_distance | DOUBLE | YES | NULL | NULL | NULL |
| fare_amount | DOUBLE | YES | NULL | NULL | NULL |
| extra | DOUBLE | YES | NULL | NULL | NULL |
| mta_tax | DOUBLE | YES | NULL | NULL | NULL |
| tip_amount | DOUBLE | YES | NULL | NULL | NULL |
| tolls_amount | DOUBLE | YES | NULL | NULL | NULL |
| ehail_fee | DOUBLE | YES | NULL | NULL | NULL |
| improvement_surcharge | DOUBLE | YES | NULL | NULL | NULL |
| total_amount | DOUBLE | YES | NULL | NULL | NULL |
| payment_type | BIGINT | YES | NULL | NULL | NULL |
| trip_type | BIGINT | YES | NULL | NULL | NULL |
| congestion_surcharge | DOUBLE | YES | NULL | NULL | NULL |
| cbd_congestion_fee | DOUBLE | YES | NULL | NULL | NULL |
| request_source | VARCHAR | YES | NULL | NULL | NULL |
| tpep_pickup_datetime | TIMESTAMP | YES | NULL | NULL | NULL |
| tpep_dropoff_datetime | TIMESTAMP | YES | NULL | NULL | NULL |
| Airport_fee | DOUBLE | YES | NULL | NULL | NULL |

### sql/exploration/04_tipos_y_muestra.sql
| VendorID | lpep_pickup_datetime | lpep_dropoff_datetime | store_and_fwd_flag | RatecodeID | PULocationID | DOLocationID | passenger_count | trip_distance | fare_amount | extra | mta_tax | tip_amount | tolls_amount | ehail_fee | improvement_surcharge | total_amount | payment_type | trip_type | congestion_surcharge | cbd_congestion_fee | request_source | tpep_pickup_datetime | tpep_dropoff_datetime | Airport_fee | filename |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2 | 2024-01-01 00:46:55 | 2024-01-01 00:58:25 | N | 1 | 236 | 239 | 1 | 1.98 | 12.8 | 1.0 | 0.5 | 3.61 | 0.0 | NULL | 1.0 | 21.66 | 1 | 1 | 2.75 | NULL | NULL | NULL | NULL | NULL | data/raw/green/2024/green_tripdata_2024-01.parquet |
| 2 | 2024-01-01 00:31:42 | 2024-01-01 00:52:34 | N | 1 | 65 | 170 | 5 | 6.54 | 30.3 | 1.0 | 0.5 | 7.11 | 0.0 | NULL | 1.0 | 42.66 | 1 | 1 | 2.75 | NULL | NULL | NULL | NULL | NULL | data/raw/green/2024/green_tripdata_2024-01.parquet |
| 2 | 2024-01-01 00:30:21 | 2024-01-01 00:49:23 | N | 1 | 74 | 262 | 1 | 3.08 | 19.8 | 1.0 | 0.5 | 3.0 | 0.0 | NULL | 1.0 | 28.05 | 1 | 1 | 2.75 | NULL | NULL | NULL | NULL | NULL | data/raw/green/2024/green_tripdata_2024-01.parquet |
| 1 | 2024-01-01 00:30:20 | 2024-01-01 00:42:12 | N | 1 | 74 | 116 | 1 | 2.4 | 14.2 | 1.0 | 1.5 | 0.0 | 0.0 | NULL | 1.0 | 16.7 | 2 | 1 | 0.0 | NULL | NULL | NULL | NULL | NULL | data/raw/green/2024/green_tripdata_2024-01.parquet |
| 2 | 2024-01-01 00:32:38 | 2024-01-01 00:43:37 | N | 1 | 74 | 243 | 1 | 5.14 | 22.6 | 1.0 | 0.5 | 6.28 | 0.0 | NULL | 1.0 | 31.38 | 1 | 1 | 0.0 | NULL | NULL | NULL | NULL | NULL | data/raw/green/2024/green_tripdata_2024-01.parquet |

## Calidad y validación incremental

In [10]:
con.execute((ROOT/'sql/eda/00_vista.sql').read_text())
for f in [ROOT/'sql/exploration/05_calidad.sql',ROOT/'sql/eda/06_validacion_anios.sql']:
    resultados[f.name]=ejecutar(f.relative_to(ROOT))
anios={f[0] for f in resultados['06_validacion_anios.sql'][0][1]}
print('2024 y 2026:',{2024,2026} <= anios,'; 2024, 2025 y 2026:',{2024,2025,2026} <= anios)

### sql/exploration/05_calidad.sql
| tipo | anio_archivo | viajes | fechas_nulas | inicio_fuera_anio | fin_antes_inicio | duracion_mayor_24h | distancia_nula | distancia_negativa | distancia_cero | importes_nulos | importes_negativos | pasajeros_nulos | pasajeros_cero | pago_nulo |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| green | 2024 | 660218 | 0 | 20 | 2 | 0 | 0 | 0 | 34574 | 0 | 2185 | 24328 | 6793 | 24328 |
| yellow | 2024 | 41169720 | 0 | 56 | 1575 | 230 | 0 | 0 | 776305 | 0 | 733885 | 4091232 | 401354 | 0 |
| green | 2025 | 591375 | 0 | 21 | 1344 | 2 | 0 | 0 | 24438 | 0 | 1776 | 49880 | 8255 | 49880 |
| yellow | 2025 | 48722602 | 0 | 29 | 2235 | 352 | 0 | 0 | 1402958 | 0 | 2853594 | 11611894 | 260062 | 0 |
| green | 2026 | 337114 | 0 | 14 | 5 | 4 | 0 | 0 | 12212 | 0 | 1025 | 48775 | 4527 | 48775 |
| yellow | 2026 | 29703355 | 0 | 17 | 10 | 263 | 0 | 0 | 952231 | 0 | 161970 | 7716688 | 91359 | 0 |

### sql/eda/06_validacion_anios.sql
| anio_archivo | tipo | archivos | viajes | primera_fecha | ultima_fecha |
| --- | --- | --- | --- | --- | --- |
| 2024 | green | 12 | 660218 | 2008-12-31 00:00:00 | 2025-01-01 22:21:15 |
| 2024 | yellow | 12 | 41169720 | 2002-12-31 16:46:07 | 2026-06-26 23:53:12 |
| 2025 | green | 12 | 591375 | 2008-12-31 15:13:04 | 2026-01-01 21:09:39 |
| 2025 | yellow | 12 | 48722602 | 2007-12-05 18:45:00 | 2025-12-31 23:59:59 |
| 2026 | green | 8 | 337114 | 2008-12-31 17:35:31 | 2026-08-31 23:58:28 |
| 2026 | yellow | 8 | 29703355 | 2001-01-01 09:23:58 | 2026-08-31 23:59:59 |

2024 y 2026: True ; 2024, 2025 y 2026: True
